In [7]:
import subprocess
import sys
import os
import signal
import time

print("Python_interpreter: ", sys.executable)

gtf_file = "/home/nlapique/Genomes/gencode.vM36.annotation.gtf"
genome_file = "/home/nlapique/Genomes/GRCm39.primary_assembly.genome.fa"
output_folder = "/home/nlapique/cassini_probe"
Python_interpreter = "/home/nlapique/.local/share/pipx/venvs/jupyterlab/bin/python"
script_dir = "/home/nlapique/cassini_scripts/"
suffix="mo"
parallel = "yes"
species="mouse"
generate_genome="yes"
genome_name = "GRCh38"
genome_anno = "gencode.vM36"
probe_type = {
    "exon_probe": "yes",
    "intron_probe": "yes",
    "junction_probe": "yes"
}

if not os.path.isdir(output_folder):
    os.makedirs(output_folder, exist_ok=True)


def run_process(command, timeout=None):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)

    try:
        stdout, stderr = process.communicate(timeout=timeout)
        print(stdout, end="")  
        print(stderr, end="")  

        if process.returncode != 0:
            print(f"Process failed with exit code {process.returncode}")
            sys.exit(1)
    except subprocess.TimeoutExpired:
        print(f"Process timed out after {timeout} seconds. Terminating...")
        process.terminate()
        try:
            process.wait(timeout=5) 
        except subprocess.TimeoutExpired:
            print("Process did not terminate. Killing...")
            process.kill()
        sys.exit(1)


print("Starting edit_gtf...")
command = [
    Python_interpreter,
    os.path.join(script_dir, "edit_gtf_v2.py"),
    "--gtf_file", gtf_file,
    "--output_folder", output_folder
]
run_process(command)


print("Starting filter_canonical_transcript...")
command = [
    Python_interpreter,
    os.path.join(script_dir, "filter_canonical_transcript_v2.py"),
    "--gtf_file", gtf_file,
    "--output_folder", output_folder
]
run_process(command)


print("Starting creates_transcript_file...")
command = [
    Python_interpreter,
    os.path.join(script_dir, "creates_transcript_file.py"),
    "--gtf_file", gtf_file,
    "--genome_file", genome_file,
    "--output_folder", output_folder
]
run_process(command)


print("Starting edit_cdna_with_exon_cds_position...")
command = [
    Python_interpreter,
    os.path.join(script_dir, "edit_cdna_with_exon_cds_position_v3.py"),
    "--gtf_file", gtf_file,
    "--genome_file", genome_file,
    "--output_folder", output_folder
]
run_process(command)


genome_dir = os.path.join(output_folder, f"{species.lower()}_genome")
if generate_genome == "yes" :
    print("Starting genome_generation...")
    
    os.makedirs(genome_dir, exist_ok=True) 
    star_command = [
        "/home/nlapique/Applications/STAR-2.7.11b/bin/Linux_x86_64_static/STAR",
        "--runThreadN", "10",
        "--runMode", "genomeGenerate",
        "--genomeDir", genome_dir,
        "--genomeFastaFiles", genome_file,
        "--sjdbGTFfile", gtf_file,
        "--sjdbOverhang", "31"
    ]
    
    
    try:
        result = subprocess.run(star_command, check=True, text=True, capture_output=True)
        print("STAR output:", result.stdout)
    except subprocess.CalledProcessError as e:
        print("STAR failed with error:", e.stderr)


print("Starting filter_gtf_before_intron...")
command = [
    Python_interpreter,
    os.path.join(script_dir, "filter_gtf_before_intron_v2.py"),
    "--gtf_file", gtf_file,
    "--output_folder", output_folder
]
run_process(command)


print("Starting Add_intron_to_gtf...")
command = [
    Python_interpreter,
    os.path.join(script_dir, "Add_intron_to_gtf.py"),
    "--gtf_file", gtf_file,
    "--output_folder", output_folder
]
run_process(command)

for param_name, param_value in probe_type.items():
    print(f"Processing {param_name} with value {param_value}")
    iej = None

    if param_name == "exon_probe" and param_value == "yes":
        iej = "exon"
    elif param_name == "intron_probe" and param_value == "yes":
        iej = "intron"
    elif param_name == "junction_probe" and param_value == "yes":
        iej = "junction"

    if iej is not None:
        output_folder_sub = output_folder + "/" + iej
        if not os.path.isdir(output_folder_sub):
            os.makedirs(output_folder_sub, exist_ok=True)
            
        
        if iej == "exon":
            print("Starting probe_generation_exon_junction_cds...")
            if parallel == "no":
                command = [
                    Python_interpreter, "-u",
                    os.path.join(script_dir, "probe_generation_exon_junction_cds_v3b.py"),
                    "--output_folder", output_folder,
                    "--output_folder_sub", output_folder_sub
                ]
                run_process(command)
            
            if parallel == "yes":
                command = [
                    Python_interpreter, "-u",
                    os.path.join(script_dir, "probe_generation_exon_junction_cds_v5b.py"),
                    "--output_folder", output_folder,
                    "--output_folder_sub", output_folder_sub
                ]
                run_process(command, timeout=1800)  
            
            
            

        if iej == "intron":
            print("Starting export_intron_sequence...")
            command = [
            Python_interpreter,
            os.path.join(script_dir, "export_intron_sequence.py"),
            "--gtf_file", gtf_file,
            "--genome_file", genome_file,
            "--output_folder", output_folder,
            "--output_folder_sub", output_folder_sub
            ]
            run_process(command)

            print("Starting probe_generation_intron...")
            if parallel == "no":
                command = [
                    Python_interpreter,
                    os.path.join(script_dir, "probe_generation_intron.py"),
                    "--output_folder_sub", output_folder_sub
                ]
                run_process(command)

            if parallel == "yes":
                command = [
                    Python_interpreter,
                    os.path.join(script_dir, "probe_generation_intron_v2.py"),
                    "--output_folder_sub", output_folder_sub
                ]
                run_process(command)

        if iej == "junction":
            print("Starting probe_generation_exon_junction...")
            if parallel == "no":
                command = [
                    Python_interpreter,
                    os.path.join(script_dir, "probe_generation_exon_junction.py"),
                    "--output_folder_sub", output_folder_sub,
                    "--output_folder", output_folder
                ]
                run_process(command)

            if parallel == "yes":
                command = [
                    Python_interpreter,
                    os.path.join(script_dir, "probe_generation_exon_junction_v2.py"),
                    "--output_folder_sub", output_folder_sub,
                    "--output_folder", output_folder
                ]
                run_process(command)
            
        
        time.sleep(15)

        
        print("Starting Add_binding_score_and _sort_exon...")
        if parallel == "no":
            command = [
                Python_interpreter, "-u",
                os.path.join(script_dir, "Add_binding_score_and_sort_exon_b.py"),
                "--output_folder_sub", output_folder_sub
            ]
            run_process (command)
        
        if parallel == "yes":
            command = [
                Python_interpreter, "-u",
                os.path.join(script_dir, "Add_binding_score_and_sort_exon_v4b.py"),
                "--output_folder_sub", output_folder_sub
            ]
            run_process(command)

            

            


        
        time.sleep(5)
        print("Starting merge_exon_csv_and_add_probe_names...")
        if param_name == "exon_probe" and param_value == "yes":
            type_iej = "e"
        elif param_name == "intron_probe" and param_value == "yes":
            type_iej = "i"
        elif param_name == "junction_probe" and param_value == "yes":
            type_iej = "j"


        
        if parallel == "no":
            command = [
                Python_interpreter,
                os.path.join(script_dir, "merge_exon_csv_and_add_probe_names_v2b.py"),
                "--output_folder_sub", output_folder_sub,
                "--suffix", suffix,
                "--type_iej",type_iej
            ]
            run_process(command)
        
        
        
        if parallel == "yes":
            command = [
                Python_interpreter,
                os.path.join(script_dir, "merge_exon_csv_and_add_probe_names_v3b.py"),
                "--output_folder_sub", output_folder_sub,
                "--suffix", suffix,
                "--type_iej",type_iej
            ]
            run_process(command)
        print("Starting trim_sequence_to_fasta_16nt...")
        time.sleep(5)
        command = [
            Python_interpreter,
            os.path.join(script_dir, "trim_sequence_to_fasta_16nt_exon_b.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        time.sleep(5)
        print("Starting trim_sequence_to_fasta_32nt...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "trim_sequence_to_fasta_32nt_exon_b.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        time.sleep(5)
        
        parent_input_dir = os.path.join(output_folder_sub, "bulk_prealignment")
        genome_dir = genome_dir 
        print(genome_dir)
        parent_output_dir = os.path.join(output_folder_sub, "bulk_star")
        script_path = os.path.join(script_dir, "align_probes_v3.sh")
        os.chmod(script_path, 0o755)
        
        print("Starting alignment...")
        result = subprocess.run(["bash", script_path, parent_input_dir, genome_dir, parent_output_dir],capture_output=True, text=True)
        time.sleep(10)
        
        
        
        parent_input_dir=parent_output_dir
        parent_output_dir=os.path.join(output_folder_sub, "bulk_sam")
        script_path = os.path.join(script_dir, "convert_sam_v2.sh")
        
        print("Starting sam conversion...")
        result = subprocess.run(["bash", script_path, parent_input_dir, parent_output_dir],capture_output=True, text=True)
        
        
        time.sleep(5)
        
        print("Starting convert_sam_filter_probes...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "convert_sam_filter_probes_v3.py"), # use convert_sam_filter_probes_v3.py if there is an error
            "--gtf_file", gtf_file,
            "--output_folder", output_folder,
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting split_reverse_forward...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "split_reverse_forward.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        
        print("Starting filter_no_match_32nt...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "filter_no_match_32nt.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting filter_rows_alignment_mismatch...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "filter_rows_alignment_mismatch.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        
        
        print("Starting Perfect_match_other_genes...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "Perfect_match_other_genes.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting 1mismatch_other_gene...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "1mismatch_other_gene.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting convert_sam_to_gene...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "convert_sam_to_gene_v2.py"), #if error use convert_sam_to_gene_v2.py
            "--gtf_file", gtf_file,
            "--output_folder", output_folder,
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting split_reverse_forward_16nt...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "split_reverse_forward_16nt.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting filter_no_match_16nt...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "filter_no_match_16nt.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting filter_rows_alignment_mismatch_loop...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "filter_rows_alignment_mismatch_loop.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        
        print("Starting Perfect_match_other_genes_16nt...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "Perfect_match_other_genes_16nt.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting Merge_trim_count...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "Merge_trim_count.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        
        
        print("Starting Merge_trim_name...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "Merge_trim_name.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
 
        
        print("Starting matching_extension...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "matching_extension.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        
        print("Starting edit_gene_name_match_length...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "edit_gene_name_match_length.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting sort_length_match...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "sort_length_match.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting merge_all_align_off_targets...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "merge_all_align_off_targets.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting add_suffixe...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "add_suffixe.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting convert_SA...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "convert_SA.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting merge_intron_exon...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "merge_intron_exon.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting specificity_score...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "specificity_score.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        
        time.sleep(20)
        
        print("Starting merge_all_intron_exon...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "merge_all_intron_exon.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        
        print("Starting sort_merged_intron_exon...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "sort_merged_intron_exon.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting Add_specificity_score_to_merged_probe...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "Add_specificity_score_to_merged_probe.py"),
            "--output_folder_sub", output_folder_sub
        ]
        run_process(command)
        
        print("Starting Add_columns_and_revcomp...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "Add_columns_and_revcomp.py"),
            "--output_folder_sub", output_folder_sub,
            "--iej", iej,
            "--species", species,
            "--genome_name", genome_name,
            "--genome_anno", genome_anno
        ]
        run_process(command)
        
        
        print("Starting final_column_arrangement_and_specificity_tag...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "final_column_arrangement_and_specificity_tag.py"),
            "--output_folder_sub", output_folder_sub,
            "--output_folder", output_folder,
            "--species", species,
            "--iej", iej
        ]
        run_process(command)
        
        print("Starting convert_gene_symbols_upper_case...")
        command = [
            Python_interpreter,
            os.path.join(script_dir, "convert_gene_symbols_upper_case.py"),
            "--output_folder", output_folder
        ]
        run_process(command)




print("Subprocesses finished.")


Python_interpreter:  /home/nlapique/.local/share/pipx/venvs/jupyterlab/bin/python
Starting edit_gtf...
Input GTF file: /home/nlapique/Genomes/gencode.vM36.annotation.gtf
Output directory: /home/nlapique/cassini_probe
Starting filter_canonical_transcript...
Filtered GTF file saved at: /home/nlapique/cassini_probe/gencode.vM36.annotation_canonical.gtf
Starting creates_transcript_file...
Starting edit_cdna_with_exon_cds_position...

Updated FASTA saved to /home/nlapique/cassini_probe/genome_filtered_transcripts_ensembl_updated.fa
Starting genome_generation...


KeyboardInterrupt: 